---
title: "Workflow - Country Ecosystem Data Preparation"
subtitle: "RLE Assessments Workshop"
author:
  - name: "Tyler Erickson"
date: "2026-07-23"
execute:
  enabled: false
  freeze: false
  cache: false
format:
  revealjs:
    theme: night
    css: styles.css
    slide-number: true
    navigation-mode: vertical
---

## Presentation Config {visibility="hidden"}

```{=html}
<style>
[data-class-output="small-output"] .cell-output table { font-size: 0.4em; }
</style>
```

## Overview

This presentation demonstrates how to prepare ecosystem map data for a country for efficient analysis.

It uses Colombia as an example.

## Source data

The 2024 ecosystem map is posted on the [IDEAM website](https://www.ideam.gov.co/) which contains a [page listing datasets related to ecosystems](https://www.ideam.gov.co/ecosistemas). 

## IDEAM Ecosystems Portal

:::: {.columns}

::: {.column width="60%"}
The [IDEAM](https://www.ideam.gov.co/) ecosystems page hosts Colombia's national ecosystem datasets.

From here you can browse and download the Map of Continental, Coastal and Marine Ecosystems (MEC) used in this workflow.

The ecosystem data is distributed as a **~2.3 GB zip** file, which contains the data in both Shapefile and Geodatabase formats. 

We will make copies in cloud-optimized formats, to allow additional workflows.
:::

::: {.column width="40%"}
[![IDEAM ecosystems portal](images/ideam_ecosystem_webpage.png){height="500"}](https://www.ideam.gov.co/ecosistemas)
:::

::::

## Local Processing {.scrollable}

Click on the [Download the shapefile of the map of ecosystems, continental, coastal and marine or Colombia (MEC)" 1:100,000 2024](https://e436.short.gy/MEcosis2024) which resolves to a login-gated Sharepoint link. Manually download the file to your local computer.

Once it downloads, update the following cell with the actual path.

In [1]:
from pathlib import Path

zip_path = Path("~/Downloads/Mapa_Ecosistemas_Continentales_Costeros_Marinos_100K_2024.zip").expanduser()

Inspect the file to determine the format.

In [2]:
#| echo: true
# Inspect the file to determine the format.
import zipfile

# Peek inside the zip without extracting it.
with zipfile.ZipFile(zip_path) as zf:
    names = zf.namelist()

print(f"{len(names)} entries in the zip; first few:")
for name in names[:5]:
    print("   ", name)

# Identify the geospatial data sources present (the zip may hold more than one).
gdbs = sorted({n[: n.index(".gdb/") + 4] for n in names if ".gdb/" in n})
shps = sorted(n for n in names if n.lower().endswith(".shp"))

print()
print("File Geodatabases:", gdbs or "none")
print("Shapefiles:       ", shps or "none")

67 entries in the zip; first few:
    Mapa_Ecosistemas_Continentales_Costeros_Marinos_100K_2024/
    Mapa_Ecosistemas_Continentales_Costeros_Marinos_100K_2024/ECOSISTEMAS_MEC_122024.gdb/
    Mapa_Ecosistemas_Continentales_Costeros_Marinos_100K_2024/ECOSISTEMAS_MEC_122024.gdb/a00000001.freelist
    Mapa_Ecosistemas_Continentales_Costeros_Marinos_100K_2024/ECOSISTEMAS_MEC_122024.gdb/a00000001.gdbindexes
    Mapa_Ecosistemas_Continentales_Costeros_Marinos_100K_2024/ECOSISTEMAS_MEC_122024.gdb/a00000001.gdbtable

File Geodatabases: ['Mapa_Ecosistemas_Continentales_Costeros_Marinos_100K_2024/ECOSISTEMAS_MEC_122024.gdb']
Shapefiles:        ['Mapa_Ecosistemas_Continentales_Costeros_Marinos_100K_2024/SHAPE/e_eccmc_100K_2024.shp']


The zip file contains the same ecosystem layer in two formats: an [Esri File Geodatabase](https://doc.esri.com/en/arcgis-pro/latest/help/data/geodatabases/manage-file-gdb/file-geodatabases.html) and a [Shapefile](https://doc.arcgis.com/en/arcgis-online/reference/shapefiles.htm). We'll read the geodatabase.

In [3]:
#| echo: true
# Build the GDAL /vsizip/ path pointing at the .gdb directory inside the zip.
gdb_source = (
    f"/vsizip/{zip_path}"
    "/Mapa_Ecosistemas_Continentales_Costeros_Marinos_100K_2024"
    "/ECOSISTEMAS_MEC_122024.gdb"
)

Read the geodatabase into a GeoDataFrame. (This may take up to 60 seconds.)

In [4]:
#| echo: true
import geopandas as gpd

gdf = gpd.read_file(gdb_source)

/Users/tylere/Documents/GitHub/RLE-Assessment/rle_workshop/.pixi/envs/default/lib/python3.14/site-packages/pyogrio/raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


Display the MEC data.

In [5]:
#| echo: true
gdf

,tipo_ecos,gra_trans,gran_bioma,bioma_preliminar,bioma_IAvH,ecos_sintesis,ecos_general,u_sintesis,amb_acuatico,subsistema,...,no_anfibio,no_aves,no_magnoliops,no_mamiferos,no_reptiles,ruleid,override,Shape_Length,Shape_Area,geometry
0,Acuatico,Natural,Pedobioma del Zonobioma Humedo Tropical,Hidrobioma,Hidrobioma Alto Caquetá,Rio,Rio de Aguas Blancas,Rio de Aguas Blancas de la zona hidrográfica C...,Lotico,Andino Atlántico,...,7.0,14.0,15.0,12.0,10.0,66,None,2.477929,0.003581,"MULTIPOLYGON (((-75.91369 0.9763, -75.91345 0...."
1,Acuatico,Transformado,Pedobioma del Zonobioma Humedo Tropical,Helobioma,Helobioma Alto Caquetá,Transicional Transformado,Transicional Transformado,Transicional Transformado en áreas que predomi...,Transicional,Andino Atlántico,...,7.0,14.0,15.0,12.0,10.0,75,None,0.028903,0.000023,"MULTIPOLYGON (((-75.52794 0.85907, -75.52701 0..."
2,Acuatico,Natural,Pedobioma del Zonobioma Humedo Tropical,Helobioma,Helobioma Alto Caquetá,Bosque,Bosque Inundable Basal,Bosque Inundable Basal en áreas que predomina ...,Transicional,Andino Atlántico,...,7.0,14.0,15.0,12.0,10.0,29,None,0.112034,0.000376,"MULTIPOLYGON (((-75.51411 0.79744, -75.514 0.7..."
3,Acuatico,Transformado,Pedobioma del Zonobioma Humedo Tropical,Helobioma,Helobioma Alto Caquetá,Transicional Transformado,Transicional Transformado,Transicional Transformado en áreas que predomi...,Transicional,Andino Atlántico,...,7.0,14.0,15.0,12.0,10.0,75,None,0.031162,0.000027,"MULTIPOLYGON (((-75.52821 0.87225, -75.5277 0...."
4,Acuatico,Transformado,Pedobioma del Zonobioma Humedo Tropical,Helobioma,Helobioma Alto Caquetá,Transicional Transformado,Transicional Transformado,Transicional Transformado en áreas que predomi...,Transicional,Andino Atlántico,...,7.0,14.0,15.0,12.0,10.0,75,None,0.038580,0.000024,"MULTIPOLYGON (((-75.52416 0.79585, -75.52418 0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
460345,Terrestre,Natural,Pedobioma del Zonobioma Humedo Tropical,Litobioma,Litobioma Serranía del Naquén,Complejos Rocosos,Complejos Rocosos de Serranias,Complejos Rocosos de Serranias en áreas que pr...,N.A.,Plataformas Antíguas,...,3.0,5.0,4.0,3.0,2.0,40,None,0.033453,0.000040,"MULTIPOLYGON (((-68.21295 1.97073, -68.2133 1...."
460346,Terrestre,Natural,Zonobioma Humedo Tropical,Zonobioma Humedo Tropical,Zonobioma Humedo Tropical Serranía del Naquén,Bosque,Bosque Basal Humedo,Bosque Basal Humedo en áreas que predomina Bos...,N.A.,Plataformas Antíguas,...,3.0,5.0,4.0,3.0,2.0,24,None,0.020580,0.000025,"MULTIPOLYGON (((-67.49606 2.17007, -67.4946 2...."
460347,Terrestre,Natural,Zonobioma Humedo Tropical,Zonobioma Humedo Tropical,Zonobioma Humedo Tropical Serranía del Naquén,Bosque,Bosque Basal Humedo,Bosque Basal Humedo en áreas que predomina Bos...,N.A.,Plataformas Antíguas,...,3.0,5.0,4.0,3.0,2.0,24,None,0.033420,0.000028,"MULTIPOLYGON (((-67.48783 2.17801, -67.48786 2..."
460348,Terrestre,Natural,Zonobioma Humedo Tropical,Zonobioma Humedo Tropical,Zonobioma Humedo Tropical Serranía del Naquén,Bosque,Bosque Basal Humedo,Bosque Basal Humedo en áreas que predomina Bos...,N.A.,Plataformas Antíguas,...,3.0,5.0,4.0,3.0,2.0,24,None,0.020843,0.000025,"MULTIPOLYGON (((-67.34087 1.98014, -67.34116 1..."


In [6]:
#| echo: true
# Summarize the number of unique values of ecosystem columns.
print(f'{gdf["ecos_sintesis"].nunique() = }')
print(f'{gdf["ecos_general"].nunique() = }')
print(f'{gdf["u_sintesis"].nunique() = }')

gdf["ecos_sintesis"].nunique() = 28
gdf["ecos_general"].nunique() = 87
gdf["u_sintesis"].nunique() = 14122


In [7]:
#| echo: true
# Sort by ecosystem so related records are grouped together.
gdf = gdf.sort_values("ecos_general", ignore_index=True)

# Write the MEC data to a local parquet file.
gdf.to_parquet("data/ECOSISTEMAS_MEC_122024.parquet", compression="zstd")

Also write a small 5 record dataset, to speed up development/testing.

In [9]:
gdf.head().to_parquet("data/ECOSISTEMAS_MEC_122024_5records.parquet", compression="zstd")

## Bogota Area Subset

Create a subset of ecoregions around Bogota that will be easier to work with.

In [10]:
from shapely.geometry import Polygon

# Area of interest around Bogota (from an Earth Engine polygon, in EPSG:4326 lon/lat).
bogota_aoi = Polygon(
    [
        [-74.27360864858895, 4.898092789377364],
        [-74.27360864858895, 4.29853018346521],
        [-73.3809694884327, 4.29853018346521],
        [-73.3809694884327, 4.898092789377364],
    ]
)

# Reproject the AOI to match the ecosystem data's CRS before the spatial query.
bogota_aoi = gpd.GeoSeries([bogota_aoi], crs="EPSG:4326").to_crs(gdf.crs).iloc[0]

# Subset to the features that intersect the AOI.
gdf_bogata = gdf[gdf.intersects(bogota_aoi)]
gdf_bogata.shape

gdf_bogata.to_parquet("data/ECOSISTEMAS_MEC_122024_bogota_area.parquet", compression="zstd")

### Publish on cloud object storage

The ecosystems table was converted to a cloud-native format (parquet) and published on Source Cooperative:

Product page: https://source.coop/tyler/colombia-ecosystems-map/ecosistemas/e_eccmc_100K_2024.parquet
Data URL: https://data.source.coop/tyler/colombia-ecosystems-map/ecosistemas/e_eccmc_100K_2024.parquet


In [ ]:
# data_url = 'https://data.source.coop/tyler/colombia-ecosystems-map/ecosistemas/ECOSISTEMAS_MEC_122024.parquet'
# data_url = 'https://data.source.coop/tyler/colombia-ecosystems-map/ecosistemas/ECOSISTEMAS_MEC_122024_5records.parquet'
data_url = 'https://data.source.coop/tyler/colombia-ecosystems-map/ecosistemas/ECOSISTEMAS_MEC_122024_bogata_area.parquet'

## Inspect the published dataset

Read directly from the cloud-hosted GeoParquet and summarize the geometry type and polygon counts.

In [ ]:
import geopandas as gpd
import fsspec

data = gpd.read_parquet(fsspec.open(data_url).open())

In [ ]:
data.shape

In [ ]:
data.geometry.crs

In [ ]:
print(data.geometry.geom_type.value_counts())                       # Polygon vs MultiPolygon


In [ ]:
data.columns

In [ ]:
from itables import show

show(data["gran_bioma"].value_counts().to_frame("count"))

In [ ]:
show(data["bioma_preliminar"].value_counts().to_frame("count"))

In [ ]:
show(data["ecos_sintesis"].value_counts().to_frame("count"))

In [ ]:
show(data["ecos_general"].value_counts().to_frame("count"))

In [ ]:
show(data["bioma_IAvH"].value_counts().to_frame("count"))

In [ ]:
show(data["u_sintesis"].value_counts().to_frame("count"))


In [ ]:
gdf.head()['ecos_general']